# Simple IK
- Set End-Effector target position & rotation
- Calculate pseudo-inverse of Jacobian
- Multiply error & execute

#### 1. Create environment

In [1]:
import os
import sys
import numpy as np
import time
import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
xml_path = '../asset/panda_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

In [3]:
""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)

while viewer.is_alive():
    mujoco.mj_forward(model, data)
    viewer.render()
    time.sleep(0.1) # for visualization stability

# close
viewer.close()
del(viewer)

#### 2. Set EE position & orientation

In [4]:
# ratio between position and rotation error 
POS_ROT_RATIO = 0.57

In [5]:
""" CURRENT PR """
site_names = get_site_names(model, data)
print(site_names) 
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
eef_site_pos = data.site_xpos[eef_site_id]
eef_site_rmat = data.site_xmat[eef_site_id]
eef_site_rmat = eef_site_rmat.reshape(3, 3) 
print(f"Current EE position: {eef_site_pos}, Euler angles: {eef_site_rmat}")

""" TARGET PR """
pos_target = np.array([0.5, -0.2, 0.3])
euler_target = np.array([0, 1.57, 0])
rmat_target = euler2rmat(euler_target)
print(f"Target EE position: {pos_target}, Rotation matrix: {rmat_target}")

['right_center', 'eef_site']
Current EE position: [ 3.92404417e-01 -6.56356925e-34  4.58585349e-01], Euler angles: [[ 9.59410820e-01  2.82012194e-01  1.57009246e-16]
 [ 2.82012194e-01 -9.59410820e-01 -9.24446373e-33]
 [ 1.50636369e-16  4.42785220e-17 -1.00000000e+00]]
Target EE position: [ 0.5 -0.2  0.3], Rotation matrix: [[ 7.96326711e-04  0.00000000e+00  9.99999683e-01]
 [ 0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [-9.99999683e-01  0.00000000e+00  7.96326711e-04]]


#### 3. Solve IK
- Calculate unit error
- Loop: jacobian pseudo inverse

In [6]:
""" CLIPPING ERROR """
POSITION_CLIPPING = 0.02
ROTATION_CLIPPING = 0.2

In [7]:
def get_ik_error_clipped(
        p_current,
        r_current,
        p_target,
        r_target,
        ):
    pos_error = p_target - p_current
    rot_error = r_target @ r_current.T
    euler_error = rmat2euler(rot_error)* POS_ROT_RATIO
    pos_error_clipped = np.clip(pos_error, -POSITION_CLIPPING, POSITION_CLIPPING)
    euler_error_clipped = np.clip(euler_error, -ROTATION_CLIPPING, ROTATION_CLIPPING)
    return pos_error_clipped, euler_error_clipped

In [8]:
""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)

mujoco.mj_resetData(model, data)
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # get current EE position & rotation
    eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
    eef_site_pos = data.site_xpos[eef_site_id]
    eef_site_rmat = data.site_xmat[eef_site_id]
    eef_site_rmat = eef_site_rmat.reshape(3, 3)
    # # calculate error 
    pos_error, euler_error = get_ik_error_clipped(
        p_current=eef_site_pos,
        r_current=eef_site_rmat,
        p_target=pos_target,
        r_target=rmat_target,
    )
    error = np.concatenate([pos_error, euler_error])

    # terminate condition
    if np.linalg.norm(pos_error) < 0.01 and np.linalg.norm(euler_error) < 0.01:
        print("Target reached!")

    # calculate jacobian
    jacobian_p, jacobian_r = get_jacobian(model, data, 'eef_site', type='site')
    jacobian = np.concatenate([jacobian_p, jacobian_r], axis=0)
    # j_inverse = get_pseudo_inverse(jacobian, method="svd", sigma_threshold=0.01)
    j_inverse = get_pseudo_inverse(jacobian, method="dls", damping=0.1)
    print(f"Jacobian inverse shape: {j_inverse.shape}, error shape: {error.shape}")
    # calculate qpos error
    qpos_error = j_inverse @ error
    # update qpos
    data.qpos += qpos_error
    mujoco.mj_forward(model, data)
    viewer.render()
    time.sleep(0.1) # for visualization stability

# close
viewer.close()
del(viewer)

Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 6), error shape: (6,)
Jacobian shape: (6, 7)
Jacobian inverse shape: (7, 